## Train pi0

In [1]:
from datetime import datetime
import os
from dotenv import load_dotenv
load_dotenv()  # loads HF_TOKEN and WANDB_API_KEY from .env
## Set NCCL environment variables for distributed training in GCP G4 
## single instance, multi-GPU setup. Adjust the interface name (e.g., "ens3") as needed for your specific instance type.
os.environ.pop("NCCL_NET", None) # on single instance, multi-GPU setup, NCCL_NET should not be set to "IB" or "TCP"
os.environ["NCCL_SOCKET_IFNAME"] = "ens3" # Adjust "ens3" to the correct network interface for your GCP instance (e.g., "ens4", "eth0", etc.)
os.environ["NCCL_P2P_LEVEL"] = "PHB" # Set the P2P level to "PHB" (PCIe Host Bridge) for optimal GPU communication on a single instance
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Disable parallelism in tokenizers to avoid potential issues with multiprocessing
import warnings
warnings.filterwarnings("ignore")

In [2]:
!rm -rf ckpt #remove checkpoint directory if it already exists to avoid conflicts with previous runs
!rm -rf logs #remove logs directory if it already exists to avoid conflicts with previous runs

In [ ]:
import configparser
config = configparser.ConfigParser()
config.read("experiment-pi0.cfg")
exp = config["experiment"]

DATASET_ROOT = exp["DATASET_ROOT"]
DATASET_REPO = exp["DATASET_REPO"]
POLICY_REPO = exp["POLICY_REPO"]
OUTPUT_DIR = exp["OUTPUT_DIR"]
JOB_NAME = exp["JOB_NAME"] + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
MAX_TRAIN_STEPS = int(exp["MAX_TRAIN_STEPS"])
CHUNK_SIZE = int(exp["CHUNK_SIZE"])
ACTION_STEPS = int(exp["ACTION_STEPS"])
BATCH_SIZE = int(exp["BATCH_SIZE"])

In [4]:
#--dataset.root={DATASET_ROOT} \
!hf download {DATASET_REPO} --repo-type dataset --local-dir {DATASET_ROOT} 

Fetching 29 files:   0%|                                 | 0/29 [00:00<?, ?it/s]Downloading 'data/chunk-000/file-004.parquet' to 'dataset/clr_teleoperation_dataset/.cache/huggingface/download/data/chunk-000/EWlqKgCh4PHdhuP-twKPGW58408=.88d4d6a133ae4e3ebc3ad344722172566779b2bd2dd74b18e9f7282b193c2bdd.incomplete'

data/chunk-000/file-004.parquet:   0%|               | 0.00/108M [00:00<?, ?B/s]Downloading 'data/chunk-000/file-000.parquet' to 'dataset/clr_teleoperation_dataset/.cache/huggingface/download/data/chunk-000/Rk5xtNxR079Ysk7EiCU6jR-aXfw=.3840f9712ae88373bc58cae38078d17f3151d02c76fee25b07f9193d16bb572d.incomplete'


data/chunk-000/file-005.parquet:   0%|               | 0.00/103M [00:00<?, ?B/s]


data/chunk-000/file-001.parquet:   0%|               | 0.00/142M [00:00<?, ?B/s]



data/chunk-000/file-002.parquet:   0%|               | 0.00/141M [00:00<?, ?B/s]




data/chunk-000/file-003.parquet:   0%|               | 0.00/128M [00:00<?, ?B/s]





data/chunk-000/file-000.parquet: 

In [ ]:
# Train the policy pi05 with the specified configuration. The model will be pushed to the Hugging Face Hub under the provided POLICY_REPO name after training.
!accelerate launch \
--multi_gpu \
--num_machines=1 \
--num_processes=4 \
--mixed_precision=bf16 \
$(which lerobot-train) \
    --dataset.repo_id={DATASET_REPO} \
    --dataset.root={DATASET_ROOT} \
    --policy.type=pi0 \
    --policy.push_to_hub=true \
    --policy.repo_id={POLICY_REPO} \
    --output_dir={OUTPUT_DIR} \
    --job_name={JOB_NAME} \
    --policy.pretrained_path=lerobot/pi05_base \
    --policy.compile_model=false \
    --policy.gradient_checkpointing=true \
    --wandb.enable=true \
    --policy.dtype=bfloat16 \
    --policy.freeze_vision_encoder=false \
    --policy.train_expert_only=false \
    --steps={MAX_TRAIN_STEPS} \
    --log_freq=50 \
    --eval_freq=-1 \
    --policy.device=cuda \
    --policy.chunk_size={CHUNK_SIZE} \
    --policy.n_action_steps={ACTION_STEPS} \
    --batch_size={BATCH_SIZE}


INFO 2026-04-03 17:29:04 ot_train.py:274 {'batch_size': 32,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         '

## Train GR00T N 1.5 TO BE DONE YET

In [ ]:
!pip install ninja "packaging>=24.2,<26.0"
!pip install peft
!pip install dm-tree==0.1.9
!pip install -U transformers
!pip install flash-attn==2.7.3 --no-build-isolation

In [ ]:
!lerobot-train\
    --dataset.repo_id=Jeongeun/tutorial_v2 \
    --dataset.root=dataset/leader_data \
    --policy.type=groot \
    --policy.repo_id=={YOUR REPO} \
    --output_dir=ckpt/tutorial_v2_groot \
    --job_name=tutorial_v2_groot \
    --wandb.enable=false \
    --steps=20000 \
    --policy.chunk_size=20 \
    --policy.n_action_steps=20 \
    --batch_size=32